# jscip Demo 2: Distributions, Units, and Parameter Banks

This notebook demonstrates the core features of `jscip`:

- Defining independent and derived parameters
- Using multiple distributions (uniform, normal, Bernoulli, custom scipy)
- Attaching optional units to parameters
- Sampling with `ParameterBank`, including vector mode
- Computing simple log-probabilities over bounds


In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from jscip import IndependentParameter, DerivedParameter, ParameterBank, ParameterSet


## 1. Basic independent and derived parameters

We start with a simple example without units:
- `length`: sampled uniformly on [0, 10]
- `width`: fixed at 2.0
- `area`: derived as `length * width`


In [2]:
length = IndependentParameter(value=5.0, is_sampled=True, range=(0.0, 10.0))
width = IndependentParameter(value=2.0, is_sampled=False)
area = DerivedParameter(lambda ps: ps['length'] * ps['width'])

bank_basic = ParameterBank(parameters={'length': length, 'width': width, 'area': area})
print(bank_basic.summary())


ParameterBank:
----------------
length: sampled, value=5.0, range=(0.0, 10.0)
width: fixed, value=2.0, range=None
area: DerivedParameter(function=<lambda>)
Constraints:
----------------


In [3]:
sample_basic = bank_basic.sample()
sample_basic


ParameterSet(length     8.642268
width      2.000000
area      17.284536
dtype: float64)

## 2. Using different distributions

Here we demonstrate several distribution options:

- Uniform over a range (default when `is_sampled=True` and `range` is provided)
- Normal distribution configured via a dict
- Bernoulli distribution for a discrete parameter
- A user-supplied frozen `scipy.stats` distribution


In [4]:
# Uniform over [0, 1] via range (backwards-compatible behavior)
u_param = IndependentParameter(value=0.5, is_sampled=True, range=(0.0, 1.0))

# Normal distribution centered at 0 with std 1
# Here we also provide a finite range so that log_prob can evaluate bounds.
n_param = IndependentParameter(
    value=0.0,
    is_sampled=True,
    range=(-5.0, 5.0),
    distribution={"kind": "normal", "loc": 0.0, "scale": 1.0},
)

# Bernoulli distribution with p=0.3, constrained to [0, 1]
b_param = IndependentParameter(
    value=0.0,
    is_sampled=True,
    range=(0.0, 1.0),
    distribution={"kind": "bernoulli", "p": 0.3},
)

# User-supplied frozen scipy.stats distribution (normal with small variance)
# Provide a reasonable range for log_prob as well.
frozen = stats.norm(loc=1.0, scale=0.1)
f_param = IndependentParameter(
    value=1.0,
    is_sampled=True,
    range=(0.0, 2.0),
    distribution=frozen,
)

dist_bank = ParameterBank(
    parameters={
        "u_param": u_param,
        "n_param": n_param,
        "b_param": b_param,
        "f_param": f_param,
    }
)
print(dist_bank.summary())

ParameterBank:
----------------
u_param: sampled, value=0.5, range=(0.0, 1.0)
n_param: sampled, value=0.0, range=(-5.0, 5.0)
b_param: sampled, value=0.0, range=(0.0, 1.0)
f_param: sampled, value=1.0, range=(0.0, 2.0)
Constraints:
----------------


In [5]:
samples_dist = dist_bank.sample(size=5)
samples_dist


,u_param,n_param,b_param,f_param
0,0.687626,1.069885,1.0,1.146425
1,0.054393,0.779366,0.0,1.131024
2,0.733897,-0.027472,0.0,0.763830
3,0.318207,-1.402252,1.0,1.089732
4,0.210793,-0.015729,0.0,0.923744


## 3. Working with units

`IndependentParameter` accepts an optional `unit` argument.

Below we use a minimal dummy unit class for illustration. In your environment,
you can swap this for a real unit from `units-llnl`.


In [9]:
from units_llnl import Unit  # or whatever units-llnl exposes

m = Unit("m")          # meters
s = Unit("s")          # seconds

In [ ]:
length_u = IndependentParameter(value=5.0, is_sampled=True, range=(0.0, 10.0), unit=m)
time_u = IndependentParameter(value=2.0, is_sampled=True, range=(1.0, 3.0), unit=s)
speed = DerivedParameter(lambda ps: ps['length_u'] / ps['time_u'])

bank_units = ParameterBank(parameters={
    'length_u': length_u,
    'time_u': time_u,
    'speed': speed,
})
print(bank_units.summary())


ParameterBank:
----------------
length_u [m]: sampled, value=5.0, range=(0.0, 10.0)
time_u [s]: sampled, value=2.0, range=(1.0, 3.0)
speed: DerivedParameter(function=<lambda>)
Constraints:
----------------


In [11]:
# Numeric defaults (no units)
defaults_numeric = bank_units.get_default_values()
defaults_numeric


ParameterSet(length_u    5.0
time_u      2.0
speed       2.5
dtype: float64)

In [12]:
# Default values with units applied on readout
defaults_with_units = bank_units.get_default_values(with_units=True)
defaults_with_units


ParameterSet(length_u    5 m
time_u      2 s
speed       2.5
dtype: object)

In [13]:
# Single sample with units
sample_with_units = bank_units.sample(with_units=True)
sample_with_units


ParameterSet(length_u    1.16831874847 m
time_u      2.81560897827 s
speed              0.414944
dtype: object)

## 4. Vector mode and log-probabilities

`ParameterBank` supports a *vector mode* where vectors contain only sampled
independent parameters in a fixed order. This is useful for optimization or
MCMC libraries that operate on plain NumPy arrays.

We also demonstrate `log_prob`, which returns 0.0 for samples within bounds
and satisfying constraints, and `-inf` otherwise.


In [14]:
# Reuse dist_bank and enable vector mode
dist_bank.vector_mode = True

# Sample a batch of vectors (only sampled independent parameters)
vectors = dist_bank.sample(size=10)
vectors.shape, vectors[:3]


((10, 4),
 array([[ 0.06750608, -0.04357651,  1.        ,  0.877981  ],
        [ 0.82613792, -0.42852952,  0.        ,  0.83933102],
        [ 0.68035476,  0.5513497 ,  0.        ,  1.04827106]]))

In [15]:
# Convert a vector back to a full ParameterSet
vec = vectors[0]
instance_from_vec = dist_bank.vector_to_instance(vec)
instance_from_vec


ParameterSet(u_param    0.067506
n_param   -0.043577
b_param    1.000000
f_param    0.877981
dtype: float64)

In [16]:
# Simple log-probability under bounds and constraints
lp = dist_bank.log_prob(instance_from_vec)
lp


np.float64(0.0)